<a href="https://colab.research.google.com/github/dhavanavijaywork/ML-2_Dhavana/blob/main/FOIL_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import pandas as pd

# 1. SAMPLE DATASET (Fraud Detection Example)
data = [
    {"ForeignLocation": True,  "HighAmount": True,  "LateNight": True,  "Fraud": True},   # Tx_101
    {"ForeignLocation": True,  "HighAmount": True,  "LateNight": False, "Fraud": True},   # Tx_102
    {"ForeignLocation": True,  "HighAmount": False, "LateNight": False, "Fraud": False},  # Tx_201
    {"ForeignLocation": False, "HighAmount": False, "LateNight": True,  "Fraud": False},  # Tx_202
]

df = pd.DataFrame(data)

# 2. FOIL GAIN CALCULATION
def foil_gain(p0, n0, p1, n1, t):
    if p1 == 0:
        return -1
    return t * (math.log2(p1 / (p1 + n1)) - math.log2(p0 / (p0 + n0)))

# 3. FOIL ALGORITHM
def FOIL(data_df, target_predicate, feature_predicates):
    pos = data_df[data_df[target_predicate] == True]
    neg = data_df[data_df[target_predicate] == False]
    learned_rules = []

    while len(pos) > 0:
        new_rule = []
        new_rule_neg = neg.copy()
        current_pos = pos.copy()

        while len(new_rule_neg) > 0:
            p0 = len(current_pos)
            n0 = len(new_rule_neg)

            best_literal = None
            best_gain = -float('inf')

            # Evaluate candidate literals
            for literal in feature_predicates:
                if literal in new_rule:
                    continue

                # Filter current sets using candidate literal
                pos_subset = current_pos[current_pos[literal] == True]
                neg_subset = new_rule_neg[new_rule_neg[literal] == True]

                p1 = len(pos_subset)
                n1 = len(neg_subset)
                t = p1

                gain = foil_gain(p0, n0, p1, n1, t)

                if gain > best_gain:
                    best_gain = gain
                    best_literal = literal

            # Add best literal to the rule
            if best_literal is None or best_gain <= 0:
                break

            new_rule.append(best_literal)
            current_pos = current_pos[current_pos[best_literal] == True]
            new_rule_neg = new_rule_neg[new_rule_neg[best_literal] == True]

        learned_rules.append(new_rule)

        # Remove positive instances covered by NewRule
        pos = pos.drop(current_pos.index)

    return learned_rules

# 4. RUN ALGORITHM
predicates = ["ForeignLocation", "HighAmount", "LateNight"]
target = "Fraud"

rules = FOIL(df, target, predicates)

# PRINT RESULTS
print("LEARNED RULES:")
for i, rule in enumerate(rules, 1):
    rule_str = " AND ".join(rule) if rule else "True"
    print(f"Rule {i}: IF {rule_str} THEN {target} = True")

LEARNED RULES:
Rule 1: IF HighAmount THEN Fraud = True
